## Iris Random Forest Classifier
End-to-end ML pipeline using SparkML, Feature Store, MLflow model versioning, and Model Serving.

**Catalog:** `ml_training` | **Schema:** `iris_classifier`


Steps:
 - Set up environment
 - Pull in Iris dataset
 - Perform EDA 
 - Preprocessing
 - Feature Engineering
 - Feature Store
 - Train/Test Split
 - Hyperparameter Tuning and Evaluation
 - Test data evaluation
 - MLFlow Experiment Tracking
 - Model Serving

In [0]:
# Configuration
CATALOG = "ml_training"
SCHEMA = "iris_classifier"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Using: {CATALOG}.{SCHEMA}")

In [0]:
import pandas as pd
from sklearn.datasets import load_iris

def load_iris_as_spark_df(spark):
    """Load the Iris dataset and convert it to a Spark DataFrame.

    Uses scikit-learn's bundled copy of the Iris dataset as a convenient
    source, then converts to Spark so the rest of the pipeline stays
    fully within SparkML.

    Args:
        spark: Active SparkSession.

    Returns:
        pyspark.sql.DataFrame: Iris data with feature columns and a
            species label column.
    """
    # Load from scikit-learn since SparkML has no built-in datasets
    iris = load_iris()

    # Build a pandas DataFrame with readable column names
    pdf = pd.DataFrame(iris.data, columns=[
        "sepal_length", "sepal_width", "petal_length", "petal_width"
    ])

    # Map numeric targets to species names for interpretability
    species_map = dict(enumerate(iris.target_names))
    pdf["species"] = pd.Series(iris.target).map(species_map)

    # Add a unique ID column -- required later for Feature Store lookups
    pdf["iris_id"] = range(len(pdf))

    return spark.createDataFrame(pdf)


iris_df = load_iris_as_spark_df(spark)
display(iris_df)

In [0]:
import numpy as np
import pandas as pd

# SparkML requires assembling features into a single vector column
from pyspark.ml.feature import VectorAssembler, StringIndexer

# Core model and pipeline components
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# SparkML uses a randomSplit method on DataFrames rather than a standalone
# train_test_split function like scikit-learn
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Optuna is used for hyperparameter tuning
import optuna

print("Dependencies loaded.")